In [0]:

from pyspark import pipelines as dp
from pyspark.sql import functions as F
import re


# ============================================================
# 1. READ CONFIGURATION
# ============================================================

def get_config(key):
    value = spark.conf.get(key, None)

    if value is None or not str(value).strip():
        raise ValueError(
            f"Missing pipeline configuration: {key}"
        )

    return str(value).strip()


source_directory = get_config("vstone.source_directory")
source_file_glob = get_config("vstone.source_file_glob")

file_format = get_config("vstone.file_format")
target_table = get_config("vstone.target_table")
table_comment = get_config("vstone.table_comment")

header = get_config("vstone.csv_header")
delimiter = get_config("vstone.csv_delimiter")
quote = get_config("vstone.csv_quote")
escape = get_config("vstone.csv_escape")
parse_mode = get_config("vstone.csv_parse_mode")

infer_column_types = get_config(
    "vstone.infer_column_types"
)

schema_evolution_mode = get_config(
    "vstone.schema_evolution_mode"
)

rescued_data_column = get_config(
    "vstone.rescued_data_column"
)

load_dt_column = get_config("vstone.load_dt_column")
source_column = get_config("vstone.source_column")


# ============================================================
# 2. CONFIGURATION VALIDATION
# ============================================================

if file_format.lower() != "csv":
    raise ValueError("Only CSV format is supported.")

if parse_mode.upper() not in [
    "PERMISSIVE",
    "FAILFAST",
    "DROPMALFORMED"
]:
    raise ValueError("Invalid CSV parsing mode.")

if load_dt_column == source_column:
    raise ValueError(
        "Audit column names must be different."
    )


# ============================================================
# 3. CLEAN COLUMN NAMES
# ============================================================

def clean_column_name(column_name):

    cleaned = re.sub(
        r"[^a-zA-Z0-9_]",
        "_",
        column_name
    )

    cleaned = re.sub(
        r"_+",
        "_",
        cleaned
    )

    cleaned = cleaned.strip("_").lower()

    if not cleaned:
        cleaned = "column"

    return cleaned


def make_unique_columns(columns, reserved_columns):

    result = []
    used = set()

    for original_name in columns:

        base_name = clean_column_name(original_name)
        cleaned_name = base_name
        counter = 1

        while (
            cleaned_name in used
            or cleaned_name in reserved_columns
        ):
            cleaned_name = f"{base_name}_{counter}"
            counter += 1

        used.add(cleaned_name)
        result.append(cleaned_name)

    return result


# ============================================================
# 4. DEFINE BRONZE STREAMING TABLE
# ============================================================

@dp.table(
    name=get_config("vstone.target_table"),
    comment=get_config("vstone.table_comment")
)
def bronze_csv_transactions():

    # --------------------------------------------------------
    # Read CSV incrementally using Auto Loader
    # --------------------------------------------------------

    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", file_format)
        .option(
            "cloudFiles.inferColumnTypes",
            infer_column_types
        )
        .option(
            "cloudFiles.schemaEvolutionMode",
            schema_evolution_mode
        )
        .option(
            "cloudFiles.rescuedDataColumn",
            rescued_data_column
        )
        .option("header", header)
        .option("delimiter", delimiter)
        .option("quote", quote)
        .option("escape", escape)
        .option("mode", parse_mode)
        .option("pathGlobFilter", source_file_glob)
        .load(source_directory)
    )

    # --------------------------------------------------------
    # Clean column names
    # --------------------------------------------------------

    if not df.columns:
        raise ValueError(
            "No CSV columns detected. Check the source CSV file."
        )

    reserved_columns = {
        clean_column_name(load_dt_column),
        clean_column_name(source_column)
    }

    cleaned_columns = make_unique_columns(
        df.columns,
        reserved_columns
    )

    df = df.toDF(*cleaned_columns)

    # --------------------------------------------------------
    # Add audit columns
    # --------------------------------------------------------

    df = (
        df
        .withColumn(
            load_dt_column,
            F.current_timestamp()
        )
        .withColumn(
            source_column,
            F.input_file_name()
        )
    )

    return df
